# NetScaler ADC y Gateway en las listas de vulnerabilidades explotadas

Cuaderno de lectura de la medición `netscaler/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda [«Cuando el aviso llega después del ataque: NetScaler y qué hacer con un equipo que ya estuvo expuesto»](https://manpla.net/posts/el-aviso-llega-despues/). Carga el fichero de al lado (o lo descarga del repositorio si se ejecuta fuera de él), muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/netscaler/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
res = json.loads(texto("resumen.json"))
print("KEV", res["kev_version"], "·", res["netscaler_en_kev"], "entradas de NetScaler ·",
      res["mismo_dia_que_publicacion"], "de", res["posteriores_carga_inicial"],
      "el mismo día de la publicación · resto:", res["dias_publicacion_a_kev_resto"])
altas = leer("altas.csv")
altas[["cve", "publicado_nvd", "alta_kev", "dias_publicacion_a_kev", "alta_eu_kev",
       "triaje_forense", "honeypot_primera", "honeypot_conexiones_1d"]]

## Una figura

In [ ]:
post = altas[altas.alta_kev > "2021-11-03"].sort_values("alta_kev").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 4.8))
colores = ["C3" if m else "C0" for m in post.mismo_dia]
ax.barh(post.index, post.dias_publicacion_a_kev, color=colores)
ax.scatter(post.index * 0, post.index, color=colores, zorder=3)
ax.set_yticks(post.index, post.cve); ax.invert_yaxis()
ax.set_xlabel("días de la publicación del CVE al alta en el KEV")
ax.set_title(f"{int(post.mismo_dia.sum())} de {len(post)} altas el mismo día de su publicación")
plt.tight_layout()